Exercise 2.7 (file extensions)
This exercise can give two points at maximum!

Part 1.

Write function file_extensions that gets as a parameter a filename. It should read through the lines from this file. Each line contains a filename. Find the extension for each filename. The function should return a pair, where the first element is a list containing all filenames with no extension (with the preceding period (.) removed). The second element of the pair is a dictionary with extensions as keys and corresponding values are lists with filenames having that extension.

Sounds a bit complicated, but hopefully the next example will clarify this. If the file contains the following lines
```
file1.txt
mydocument.pdf
file2.txt
archive.tar.gz
test
```
then the return value should be the pair: (["test"], { "txt" : ["file1.txt", "file2.txt"], "pdf" : ["mydocument.pdf"], "gz" : ["archive.tar.gz"] } )

Part 2.

Write a main method that calls the file_extensions function with "src/filenames.txt" as the argument. Then print the results so that for each extension there is a line consisting of the extension and the number of files with that extension. The first line of the output should give the number of files without extensions.

With the example in part 1, the output should be
```
1 files with no extension
gz 1
pdf 1
txt 2

```
Had there been no filenames without extension then the first line would have been 0 files with no extension. In the printout list the extensions in alphabetical order.

In [ ]:
#!/usr/bin/env python3

def file_extensions(filename):
    """This function takes a filename as input and returns a tuple containing two elements:
    1. A list of files without extensions.
    2. A dictionary where the keys are file extensions and the values are lists of files with that extension.
    """

    file_with_extensions = {}
    file_without_extensions = []

    with open(filename, 'r') as f:
         for line in f:
            if line.strip() == '':
                return (file_without_extensions, file_with_extensions)
            if '.' in line:
                file_with_extensions.setdefault(line.strip().split('.')[-1], []).append(line.strip())
            else:
                file_without_extensions.append(line.strip())

    return (file_without_extensions, file_with_extensions)

def main():
    a,b  = file_extensions('src/filenames.txt')
    print(f"{len(a)} files with no extension")
    for key, value in b.items():
        print(f"{key} {len(value)}")

if __name__ == "__main__":
    main()


## Yes — Several Ways to Clean This Up

Good working logic! Let's tighten it using tools built exactly for this kind of task.

---

### Issue 1 — Duplicated `return` Statements

You have the same `return (file_without_extensions, file_with_extensions)` written **twice** — once inside the loop (early exit on blank line), once after. Duplication like this is a maintenance risk: change one, forget the other, and they drift out of sync. Use `break` instead — one clean exit point:

```python
for line in f:
    if line.strip() == '':
        break               # stop reading, fall through to the single return below
    ...

return (file_without_extensions, file_with_extensions)
```

---

### Issue 2 — Repeated `.strip()` Calls

```python
line.strip().split('.')[-1]
...
file_with_extensions[...].append(line.strip())
```

`.strip()` is called on the **same line** multiple times. Strip once at the top of the loop, reuse the result:

```python
for line in f:
    line = line.strip()
    if line == '':
        break
    ...
```

---

### Issue 3 — `os.path.splitext()` Is the Built-in for This Exact Job

Python has a dedicated function for splitting a filename into "name" and "extension" — no manual `'.' in line` check or `.split('.')` needed:

```python
import os

os.path.splitext("photo.jpg")      # → ('photo', '.jpg')
os.path.splitext("archive.tar.gz") # → ('archive.tar', '.gz')   ← only splits the LAST dot
os.path.splitext("README")         # → ('README', '')            ← no extension
os.path.splitext(".gitignore")     # → ('.gitignore', '')        ← hidden file, correctly no ext!
```

Notice the last one — your manual `'.' in line` approach would have **incorrectly** treated `.gitignore` as having extension `gitignore` (since it contains a dot). `os.path.splitext` correctly recognizes it as a file with **no** extension — a real edge case your version silently gets wrong.

---

### Full Elegant Version

```python
#!/usr/bin/env python3

import os
from collections import defaultdict


def file_extensions(filename):
    """Categorize filenames from a file by their extension.

    Reads filenames (one per line) until a blank line or end of file.

    Returns:
        A tuple (no_extension, by_extension) where no_extension is a
        list of filenames with no extension, and by_extension is a
        dict mapping each extension (without the leading dot) to a
        list of filenames with that extension.
    """
    no_extension = []
    by_extension = defaultdict(list)

    with open(filename) as f:
        for line in f:
            line = line.strip()
            if not line:
                break

            name, ext = os.path.splitext(line)
            if ext:
                by_extension[ext.lstrip('.')].append(line)
            else:
                no_extension.append(line)

    return no_extension, dict(by_extension)


def main():
    no_ext, by_ext = file_extensions('src/filenames.txt')
    print(f"{len(no_ext)} files with no extension")
    for ext, files in by_ext.items():
        print(f"{ext} {len(files)}")


if __name__ == "__main__":
    main()
```

---

### What Changed and Why

| Change | Reason |
|---|---|
| `os.path.splitext()` instead of manual `.split('.')` | correctly handles `.gitignore`, `archive.tar.gz`, no-dot files — a real built-in designed for exactly this |
| `defaultdict(list)` instead of `setdefault(k, [])` | no repeated `, []` — cleaner when appending many times in a loop |
| `line = line.strip()` once at the top | avoids calling `.strip()` on the same string three separate times |
| `break` instead of duplicated `return` | single exit point — nothing to keep in sync |
| `if not line:` instead of `if line.strip() == '':'` | `not line` is the idiomatic emptiness check (an empty string is falsy) |
| `ext.lstrip('.')` | `splitext` includes the dot (`.jpg`) — strip it off to match your original key style (`jpg`, no dot) |

---

### Verifying Against Your Original Behavior

```python
# Your version treats '.gitignore' as extension 'gitignore':
'.gitignore'.split('.')[-1]   # → 'gitignore'   ⚠ probably not what you want

# The elegant version correctly treats it as NO extension:
os.path.splitext('.gitignore')   # → ('.gitignore', '')   ✓
```

If your actual `filenames.txt` doesn't contain hidden-file-style names, this distinction won't matter in practice — but it's a correctness improvement worth having for free.

---

### The One-Sentence Summary

> `os.path.splitext()` is the standard library's dedicated tool for separating a filename from its extension — it correctly handles edge cases (hidden files, multi-dot names) that a manual `'.' in line` + `.split('.')` approach gets wrong. Combined with `defaultdict(list)` for the grouping and a single `break` instead of duplicated returns, the logic becomes both more correct and less repetitive. 🎯

In [ ]:
#!/usr/bin/env python3

import os
from collections import defaultdict

def file_extensions(filename):
    """Categorize filenames from a file by their extension.

    Reads filenames (one per line) until a blank line or end of file.

    Returns:
        A tuple (no_extension, by_extension) where no_extension is a
        list of filenames with no extension, and by_extension is a
        dict mapping each extension (without the leading dot) to a
        list of filenames with that extension.
    """

    no_extension = []
    by_extension = defaultdict(list)

    with open(filename, 'r') as f:
         for line in f:
            line = line.strip()
            if not line:
                break

            name, ext = os.path.splitext(line)
            if ext:
                by_extension[ext.lstrip('.')].append(line)
            else:
                no_extension.append(line)

    return (no_extension, dict(by_extension))

def main():
    no_ext, by_ext = file_extensions('src/filenames.txt')
    print(f"{len(no_ext)} files with no extension")
    for ext, files in by_ext.items():
        print(f"{ext} {len(files)}")


if __name__ == "__main__":
    main()

Can you simply explain followings
```
by_extension = defaultdict(list)
dict(by_extension)```

## Two Lines, Explained Simply

---

### `by_extension = defaultdict(list)`

This creates an **empty dictionary with a superpower**: whenever you access a key that doesn't exist yet, it **automatically creates it** with an empty list — instead of raising a `KeyError`.

```python
by_extension = defaultdict(list)

print(by_extension)              # → defaultdict(<class 'list'>, {})   — empty for now

by_extension["jpg"].append("photo.jpg")
print(by_extension)              # → defaultdict(<class 'list'>, {'jpg': ['photo.jpg']})
```

**What just happened:** `"jpg"` didn't exist yet. Normally this would crash:

```python
d = {}
d["jpg"].append("photo.jpg")     # ✗ KeyError: 'jpg'
```

But because `by_extension` is a `defaultdict(list)`, accessing the missing key `"jpg"` **silently creates it** with an empty list `[]` first — **then** `.append()` runs successfully on that fresh list.

**The `list` argument is the "recipe"** — it tells the defaultdict: *"whenever you need to invent a value for a missing key, call `list()` to make it"* (which produces `[]`).

```python
by_extension["txt"].append("readme.txt")   # 'txt' also missing → auto-created
by_extension["jpg"].append("cat.jpg")       # 'jpg' EXISTS now → just appends
```

```python
print(by_extension)
# → defaultdict(<class 'list'>, {'jpg': ['photo.jpg', 'cat.jpg'], 'txt': ['readme.txt']})
```

This is exactly why your `file_extensions` function can do `by_extension[ext].append(line)` for **every** file, without ever checking *"does this extension already have a list?"* — the defaultdict handles that check invisibly.

---

### `dict(by_extension)`

This **converts** the `defaultdict` into a **plain, ordinary `dict`**.

```python
by_extension = defaultdict(list)
by_extension["jpg"].append("photo.jpg")

plain = dict(by_extension)

print(by_extension)   # → defaultdict(<class 'list'>, {'jpg': ['photo.jpg']})
print(plain)           # → {'jpg': ['photo.jpg']}
```

Notice — the **contents** are identical (same keys, same lists inside), but the **type label** changes:

```python
print(type(by_extension))   # → <class 'collections.defaultdict'>
print(type(plain))           # → <class 'dict'>
```

---

### Why Bother Converting Back?

Because the "auto-create missing keys" superpower is **dangerous** once you're done building the dict — it can cause silent, hard-to-spot bugs if someone (or some later code) checks a key that doesn't exist:

```python
by_extension = defaultdict(list)
by_extension["jpg"].append("photo.jpg")

# Later, someone innocently checks:
if by_extension["png"]:      # ⚠ this SILENTLY CREATES 'png': [] !
    print("has png files")
```

```python
print(by_extension)
# → defaultdict(..., {'jpg': [...], 'png': []})   ← 'png' now exists, unintentionally!
```

Converting to a **plain `dict`** before returning removes this risk — anyone using your function's result gets an ordinary, predictable dictionary where checking a missing key behaves normally:

```python
plain = dict(by_extension)
plain["png"]        # ✗ KeyError — as expected, no silent side effect
```

---

### The Pattern in Your Function

```python
def file_extensions(filename):
    by_extension = defaultdict(list)     # build it up — convenience mode ON

    ... # loop, appending freely without worrying about missing keys

    return no_extension, dict(by_extension)   # hand back plain dict — convenience mode OFF
```

**The idea:** use `defaultdict` **internally**, while you're actively building the structure (where the auto-create behavior is genuinely helpful) — then convert to a plain `dict` **at the boundary**, right before returning it to whoever calls your function (where the auto-create behavior would just be a hidden trap).

---

### The One-Sentence Summary

> `defaultdict(list)` creates a dictionary that auto-generates an empty list for any missing key the moment you touch it — perfect for building up grouped data without manual existence checks. `dict(by_extension)` then strips away that auto-create behavior, converting it back into an ordinary, predictable dictionary — the standard move once you're done building and ready to hand the result off. 🎯